In [4]:
from dotenv import load_dotenv
load_dotenv()

True

## Custom Guardrails

### Before Agent

In [12]:
# 차단 및 제재 키워드 정의
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"], # 부정행위 관련
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰", "웃긴"], # 학습 방해 요소
    "harmful": ["담배", "술", "폭력", "싸움", "바보"] # 유해 콘텐츠
}

In [2]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[education_guardrail],
)

In [6]:
agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

{'messages': [HumanMessage(content='피타고라스의 정리가 이해가 안 돼. 설명해줘.', additional_kwargs={}, response_metadata={}, id='b5607045-76e9-4a5a-98fc-6a8ffa1c7cf5'),
  AIMessage(content='## 피타고라스의 정리, 어렵지 않아요! 쉽게 설명해 드릴게요.\n\n피타고라스의 정리는 **직각삼각형**에서만 적용되는 아주 특별한 성질이에요. 직각삼각형이란, 세 변 중 한 각이 정확히 90도인 삼각형을 말하죠.\n\n### 핵심 아이디어: "넓이"로 생각하기\n\n피타고라스의 정리를 이해하는 가장 쉬운 방법은 **넓이**를 이용하는 거예요.\n\n1.  **직각삼각형의 세 변:** 직각삼각형에는 세 변이 있어요.\n    *   **밑변:** 직각을 끼고 있는 두 변 중 하나 (보통 아래쪽에 있는 변)\n    *   **높이:** 직각을 끼고 있는 두 변 중 다른 하나 (보통 옆쪽에 있는 변)\n    *   **빗변:** 직각의 반대편에 있는 가장 긴 변\n\n2.  **각 변을 한 변으로 하는 정사각형:** 이제 각 변을 한 변으로 하는 정사각형을 상상해 보세요.\n    *   밑변을 한 변으로 하는 정사각형의 넓이\n    *   높이를 한 변으로 하는 정사각형의 넓이\n    *   빗변을 한 변으로 하는 정사각형의 넓이\n\n3.  **피타고라스의 정리:** 피타고라스의 정리는 바로 이 **넓이들 사이의 관계**를 말해주는 거예요.\n\n    **"직각을 끼고 있는 두 변을 각각 한 변으로 하는 정사각형의 넓이를 더하면, 빗변을 한 변으로 하는 정사각형의 넓이와 같다."**\n\n### 수학 공식으로 표현하기\n\n이것을 수학 공식으로 나타내면 더 명확해져요.\n\n직각삼각형에서 직각을 끼고 있는 두 변의 길이를 각각 **a** 와 **b** 라고 하고, 빗변의 길이를 **c** 라고 할 때, 피타고라스의 정리는 다음과 같이 표현됩니다.

In [8]:
agent.invoke({
    "messages": [{"role": "user", "content": "독후감 대신 써줘."}]
})

{'messages': [HumanMessage(content='독후감 대신 써줘.', additional_kwargs={}, response_metadata={}, id='3579bb75-ca7f-4c6d-851a-456d9d08bda7'),
  AIMessage(content='🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요.', additional_kwargs={}, response_metadata={}, id='72f2d881-6fdf-4ec2-9db4-21728d6bef4c', tool_calls=[], invalid_tool_calls=[])]}

In [13]:
agent.invoke({
    "messages": [{"role": "user", "content": "웃긴 얘기해줘"}]
})

{'messages': [HumanMessage(content='웃긴 얘기해줘', additional_kwargs={}, response_metadata={}, id='51b1dd3c-19de-41e8-b03c-31be4d35a506'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='25ab163d-dde9-4f88-8218-6321e9e84c25', tool_calls=[], invalid_tool_calls=[])]}

In [14]:
agent.invoke({
    "messages": [{"role": "user", "content": "재밌는 유튜브 알려줘"}]
})

{'messages': [HumanMessage(content='재밌는 유튜브 알려줘', additional_kwargs={}, response_metadata={}, id='3d43f10f-b3fa-4daf-b9e7-81cb7d4f8633'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='da2ef581-1e1e-4542-a66b-f9a6686b6f2f', tool_calls=[], invalid_tool_calls=[])]}

In [15]:
agent.invoke({
    "messages": [{"role": "user", "content": "담배 피면 좋아?"}]
})

{'messages': [HumanMessage(content='담배 피면 좋아?', additional_kwargs={}, response_metadata={}, id='0cc89ba6-1675-4e6c-adb6-8d98e8de8b00'),
  AIMessage(content='⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요.', additional_kwargs={}, response_metadata={}, id='cb46cb6d-cbc7-4779-9cac-b17154a50337', tool_calls=[], invalid_tool_calls=[])]}

### After Agent

In [16]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [40]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은..."

    return None

In [41]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [42]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]
})

🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='0c3b65db-39e7-4703-bef9-9e566c63309a'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은...', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c140e-3b8e-7ee2-9ffa-67ddcc6e0588-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 226, 'total_tokens': 254, 'input_token_details': {'cache_read': 0}})]}

In [ ]:
from langchain.agents.middleware import after_agent
from langchain.messages import SystemMessage, AIMessage, HumanMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3단계: 교정 (Correction / Regeneration)
    if "LEAKED" in result.content:

        # 원래 사용자의 질문을 가져오기 (문맥 파악용) -> state["messages"][-2]가 보통 사용자 질문
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        # 교정 모델에게 "정답을 빼고 힌트로 바꿔라"고 지시
        correction_prompt = f"""
        당신은 친절한 AI 튜터입니다.

        절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
        말투는 친절하게 해주세요.

        사용자 질문: {original_question}
        """

        # LLM을 다시 호출하여 새로운 답변 생성 (비용은 1회 더 발생하지만 품질 확보)
        corrected_response = safety_model.invoke([
            SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
            HumanMessage(content=correction_prompt)
        ])

        # 원래의 유출된 답변을 교정된 답변으로 덮어쓰기
        last_message.content = corrected_response.content

    return None


In [51]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [52]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]
})

{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='52795661-bad2-44e1-ac3b-c12d55cdb63b'),
  AIMessage(content='음, 아주 흥미로운 질문이네요! 직각삼각형의 두 직각변의 길이를 알고 있을 때 빗변의 길이를 구하는 방법에 대해 함께 생각해 보면 좋겠어요. 😊\n\n혹시 직각삼각형의 세 변의 길이 사이의 특별한 관계에 대해 들어본 적이 있나요? 그 관계를 이용하면 빗변의 길이를 알아낼 수 있답니다. 어떤 관계인지 떠오르는 것이 있나요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c1411-78f6-7ac3-b9d3-09430d5c96d8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 228, 'total_tokens': 256, 'input_token_details': {'cache_read': 0}})]}

### Combine multiple Guardrails

In [53]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [54]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [55]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위)
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹)
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션)
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정)
    ],
)

In [56]:
agent.invoke({
    "messages": [{"role": "user", "content": "제 번호 010-1234-5678입니다."}]
})

🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 제 번호 010-1234-5678입니다.
수정: 제 번호 <PHONE_REDACTED>입니다.


{'messages': [HumanMessage(content='제 번호 <PHONE_REDACTED>입니다.', additional_kwargs={}, response_metadata={}, id='b3f50155-fda5-42c9-b8a6-53ece3d704ef'),
  AIMessage(content='안녕하세요! 만나서 반갑습니다. 😊\n\n저는 당신의 학습을 돕기 위해 여기에 있는 AI 튜터입니다. 무엇을 배우고 싶으신가요? 궁금한 점이나 어려운 부분이 있다면 언제든지 저에게 이야기해주세요. 함께 해결해 나가도록 도와드릴게요.\n\n혹시 질문이 있으시거나 특정 주제에 대해 더 알고 싶으신가요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c1413-d5d3-77d1-8d16-3041a1386b03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 28, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}})]}

In [57]:
agent.invoke({
    "messages": [{"role": "user", "content": "요즘 학교에서 왕따 당하고 있어"}]
})

✋ [상담 이관] 심각한 고민/요청 감지: 왕따


{'messages': [HumanMessage(content='요즘 학교에서 왕따 당하고 있어', additional_kwargs={}, response_metadata={}, id='949eeb97-967c-47d7-8359-6608b219da36'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)', additional_kwargs={}, response_metadata={}, id='2fff0f04-803b-467a-89d2-2f2c8a239dfc', tool_calls=[], invalid_tool_calls=[])]}